# Clasificacion de paneles solares

Pipeline unico y editable: configuracion, datos, arquitectura, entrenamiento, evaluacion y comparacion.
Modificar capas en la funcion `build_model` y parametros en la seccion CONFIGURACION.

In [ ]:
!pip -q install scikit-learn pandas matplotlib
from pathlib import Path
import json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
REPO_DIR = Path('/content/procesamiento-de-imagenes')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ana-castillo-zuain/procesamiento-de-imagenes.git', str(REPO_DIR)], check=True)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## CONFIGURACION EDITABLE

`random` entrena desde cero; `head` usa ImageNet con extractor congelado; `fine_tune` entrena la cabeza y luego descongela el extractor.

In [ ]:
DATASET_DIR = REPO_DIR / 'data' / 'yolo'
RESULTS_DIR = REPO_DIR / 'CLASIFICACION' / 'results'
ARCHITECTURES = ['efficientnetb0', 'mobilenetv2']
# El TP requiere pesos aleatorios, cabeza congelada y fine-tuning.
STRATEGIES = ['random', 'head', 'fine_tune']
SPLITS = ['solarpanels_val', 'geo_kingston_val', 'PV01_val']
IMAGE_SIZE, BATCH_SIZE = 224, 32
# Pocas epocas iniciales: EarlyStopping corta antes si la validacion deja de mejorar.
HEAD_EPOCHS, FINE_TUNE_EPOCHS = 8, 8
LEARNING_RATE, FINE_TUNE_LEARNING_RATE = 1e-3, 1e-5
DROPOUT, PATIENCE, THRESHOLD, SEED = 0.3, 3, 0.5, 42
USE_CLASS_WEIGHT = True
# Primero probar una sola combinacion; luego poner False para la grilla completa.
QUICK_TEST = True
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
if QUICK_TEST:
    # Smoke test barato y representativo: ImageNet + cabeza congelada.
    ARCHITECTURES = ['efficientnetb0']
    STRATEGIES = ['head']
    SPLITS = ['solarpanels_val']
print('Experimentos:', len(ARCHITECTURES) * len(STRATEGIES) * len(SPLITS))

## DATOS

Una imagen es positiva si su etiqueta YOLO contiene una caja de clase 0. Los splits ya están separados por fuente.

In [ ]:
def samples(split, part):
    image_dir = DATASET_DIR / split / 'images' / part
    label_dir = DATASET_DIR / split / 'labels' / part
    output = []
    for image in sorted(image_dir.glob('*.png')):
        label_file = label_dir / f'{image.stem}.txt'
        lines = label_file.read_text(encoding='utf-8').splitlines()
        positive = any(line.split() and int(float(line.split()[0])) == 0 for line in lines)
        output.append((image, int(positive)))
    if not output:
        raise ValueError(f'Split vacio: {split}/{part}')
    return output

def dataset(items, training=False):
    paths = tf.constant([str(path) for path, _ in items])
    labels = tf.constant([label for _, label in items], dtype=tf.float32)
    def load(path, label):
        image = tf.image.decode_png(tf.io.read_file(path), channels=3)
        return tf.image.resize(tf.cast(image, tf.float32), [IMAGE_SIZE, IMAGE_SIZE]), label
    data = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        data = data.shuffle(len(items), seed=SEED)
    return data.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

def class_weights(items):
    counts = np.bincount([label for _, label in items], minlength=2)
    total = counts.sum()
    return {i: float(total / (2 * count)) for i, count in enumerate(counts) if count}

display(pd.DataFrame([{'split': s, 'train': len(samples(s, 'train')), 'val': len(samples(s, 'val')),
                      'train_positive': sum(y for _, y in samples(s, 'train')),
                      'val_positive': sum(y for _, y in samples(s, 'val'))} for s in SPLITS]))

## ARQUITECTURA EDITABLE

Modificar la cabeza después de `base(x)`: agregar capas densas, BatchNormalization, Dropout o regularización.

In [ ]:
def build_model(name, strategy):
    options = {'efficientnetb0': (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
               'mobilenetv2': (keras.applications.MobileNetV2, keras.applications.mobilenet_v2.preprocess_input)}
    constructor, preprocess = options[name]
    weights = None if strategy == 'random' else 'imagenet'
    base = constructor(include_top=False, weights=weights, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name=f'{name}_base')
    base.trainable = strategy == 'random'
    inputs = keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3), name='image')
    x = keras.layers.RandomFlip('horizontal_and_vertical', name='augmentation')(inputs)
    x = keras.layers.Lambda(preprocess, name='preprocess')(x)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D(name='pooling')(x)
    x = keras.layers.Dropout(DROPOUT, name='dropout')(x)
    outputs = keras.layers.Dense(1, activation='sigmoid', name='classifier')(x)
    return keras.Model(inputs, outputs, name=f'{name}_{strategy}'), base

def compile_model(model, rate):
    model.compile(optimizer=keras.optimizers.Adam(rate), loss='binary_crossentropy', metrics=['accuracy'])

def show_layers(model):
    display(pd.DataFrame([{'layer': x.name, 'trainable': x.trainable, 'parameters': x.count_params()} for x in model.layers]))

## ENTRENAMIENTO Y EVALUACION

Cada experimento imprime capas entrenables, curvas y matriz de confusion. Fine-tuning recompila con learning rate bajo.

In [ ]:
def run_one(architecture, strategy, split):
    train_items, val_items = samples(split, 'train'), samples(split, 'val')
    train_data, val_data = dataset(train_items, True), dataset(val_items, False)
    folder = RESULTS_DIR / f'{architecture}_{strategy}_{split}'
    folder.mkdir(parents=True, exist_ok=True)
    model, base = build_model(architecture, 'head' if strategy == 'fine_tune' else strategy)
    compile_model(model, LEARNING_RATE)
    print(f'\n===== {architecture} | {strategy} | {split} =====')
    show_layers(model)
    callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
                 keras.callbacks.ModelCheckpoint(folder / 'best.keras', monitor='val_loss', save_best_only=True)]
    weights = class_weights(train_items) if USE_CLASS_WEIGHT else None
    print('Pesos de clase:', weights)
    history = model.fit(train_data, validation_data=val_data, epochs=HEAD_EPOCHS,
                        callbacks=callbacks, class_weight=weights, verbose=1)
    if strategy == 'fine_tune':
        base.trainable = True
        compile_model(model, FINE_TUNE_LEARNING_RATE)
        print('Extractor descongelado')
        show_layers(model)
        second = model.fit(train_data, validation_data=val_data, epochs=FINE_TUNE_EPOCHS,
                           callbacks=callbacks, class_weight=weights, verbose=1)
        history.history.update({f'fine_tune_{k}': v for k, v in second.history.items()})
    labels = [y for _, y in val_items]
    predictions = (model.predict(val_data, verbose=0).reshape(-1) >= THRESHOLD).astype(int)
    matrix = confusion_matrix(labels, predictions)
    result = {'architecture': architecture, 'strategy': strategy, 'split': split,
              'accuracy': accuracy_score(labels, predictions), 'precision': precision_score(labels, predictions, zero_division=0),
              'recall': recall_score(labels, predictions, zero_division=0), 'f1': f1_score(labels, predictions, zero_division=0),
              'confusion_matrix': matrix.tolist()}
    (folder / 'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    (folder / 'history.json').write_text(json.dumps(history.history, indent=2), encoding='utf-8')
    pd.DataFrame(history.history).plot(figsize=(12, 4), title=f'{architecture} / {strategy} / {split}')
    plt.show()
    ConfusionMatrixDisplay(matrix, display_labels=['sin panel', 'con panel']).plot(cmap='Blues')
    plt.show()
    return result

results = [run_one(a, t, s) for a in ARCHITECTURES for t in STRATEGIES for s in SPLITS]
results = pd.DataFrame(results)
results.to_csv(RESULTS_DIR / 'metrics.csv', index=False)
display(results)

In [ ]:
summary = results.groupby(['architecture', 'strategy'])[['accuracy', 'precision', 'recall', 'f1']].mean().round(3).reset_index()
display(summary.sort_values('f1', ascending=False))
results['model'] = results['architecture'] + ' / ' + results['strategy']
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
results.pivot(index='split', columns='model', values='f1').plot(kind='bar', ax=axes[0], title='F1 por fuente de validacion')
results.groupby('model')[['accuracy', 'precision', 'recall', 'f1']].mean().plot(kind='bar', ax=axes[1], title='Metricas promedio')
axes[0].tick_params(axis='x', rotation=0)
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()